## 1. 텍스트 임베딩의 개념

청킹을 통해 문서를 작은 단위로 분할하는 방법을 학습했습니다. 이제 이 청크들을 **검색 가능한 형태** 로 만들어야 합니다.

### 임베딩이란?

**임베딩(Embedding)** 은 텍스트를 **고차원 수치 벡터** 로 변환하는 과정입니다. 변환된 벡터는 텍스트의 **의미적 정보** 를 수치로 인코딩합니다.

```
텍스트                          벡터 (고차원)
──────────────────────────────────────────────────
"인공지능은 미래 기술이다"  →  [0.023, -0.015, 0.078, ..., 0.041]  (1536차원)
"AI는 혁신적 기술이다"     →  [0.021, -0.012, 0.075, ..., 0.039]  (유사한 벡터)
"오늘 날씨가 좋다"         →  [-0.045, 0.032, -0.018, ..., 0.063] (다른 벡터)
```

### 핵심 특성

- **의미적으로 유사한 텍스트** 는 벡터 공간에서 **가까이** 위치한다
- **의미적으로 다른 텍스트** 는 벡터 공간에서 **멀리** 위치한다
- 이를 통해 **키워드 매칭이 아닌 의미 기반 검색** 이 가능해진다

## 2. OpenAI Embedding API 활용

OpenAI의 `text-embedding-3-small` 모델을 사용하여 텍스트를 벡터로 변환합니다.

In [1]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def get_embedding(text, model = 'text-embedding-3-small'):
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

# 다양한 주제의 텍스트로 임베딩 테스트
texts = [
    "인공지능은 인간의 학습 능력을 컴퓨터로 구현한 기술이다.",
    "머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야이다.",
    "오늘 서울의 날씨는 맑고 기온은 25도이다.",
    "파이썬은 데이터 과학에 널리 사용되는 프로그래밍 언어이다.",
]

for text in texts:
    print(f'\n테스트 : {text}')
    print(f'벡터차원 : {len(get_embedding(text))}')
    print(f'벡터 앞 5개 값을 : {get_embedding(text)[:5]}')





테스트 : 인공지능은 인간의 학습 능력을 컴퓨터로 구현한 기술이다.
벡터차원 : 1536
벡터 앞 5개 값을 : [-0.0260009765625, 0.0192413330078125, -0.031707763671875, 0.0108489990234375, 0.019989013671875]

테스트 : 머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야이다.
벡터차원 : 1536
벡터 앞 5개 값을 : [-0.03070068359375, 0.0027408599853515625, 0.0280303955078125, 0.01346588134765625, 0.069580078125]

테스트 : 오늘 서울의 날씨는 맑고 기온은 25도이다.
벡터차원 : 1536
벡터 앞 5개 값을 : [0.0180206298828125, -0.01078033447265625, -0.06939697265625, -0.03668212890625, 0.023529052734375]

테스트 : 파이썬은 데이터 과학에 널리 사용되는 프로그래밍 언어이다.
벡터차원 : 1536
벡터 앞 5개 값을 : [0.0020313262939453125, 0.06939697265625, -0.01097869873046875, -0.0006170272827148438, 0.07879638671875]


## 3. 코사인 유사도와 벡터 검색

두 벡터 간의 **유사도**를 측정하는 대표적인 방법이 **코사인 유사도(Cosine Similarity)** 입니다. 값이 1에 가까울수록 유사하고, 0에 가까울수록 무관합니다.

In [2]:
embeddings = [ get_embedding(t) for t in texts]
def cosine_similarity(vec_a, vec_b):
    a = np.array(vec_a)
    b = np.array(vec_b)
    return float(np.dot(a,b)/(np.linalg.norm(a) * np.linalg.norm(b)))

# 헤더 출력
print(f"{'':>8}", end="")
for j in range(len(texts)):
    print(f"텍스트{j+1:>2}", end="  ")
print()

for i in range(len(texts)):
    print(f"텍스트{i+1}", end=" ")
    for j in range(len(texts)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"  {sim:.4f}", end="")
    print()

print()
print("[텍스트 목록]")
for i, t in enumerate(texts):
    print(f"  텍스트{i+1}: {t}")

print()
print("[유사도 해석]")
sim_12 = cosine_similarity(embeddings[0], embeddings[1])
sim_13 = cosine_similarity(embeddings[0], embeddings[2])

print(f"  AI-머신러닝 유사도: {sim_12:.4f}")
print(f"  AI-날씨 유사도:     {sim_13:.4f}")

        텍스트 1  텍스트 2  텍스트 3  텍스트 4  
텍스트1   1.0000  0.4824  0.0036  0.2905
텍스트2   0.4824  1.0000  0.0395  0.3349
텍스트3   0.0036  0.0395  1.0000  0.0920
텍스트4   0.2905  0.3349  0.0920  1.0000

[텍스트 목록]
  텍스트1: 인공지능은 인간의 학습 능력을 컴퓨터로 구현한 기술이다.
  텍스트2: 머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야이다.
  텍스트3: 오늘 서울의 날씨는 맑고 기온은 25도이다.
  텍스트4: 파이썬은 데이터 과학에 널리 사용되는 프로그래밍 언어이다.

[유사도 해석]
  AI-머신러닝 유사도: 0.4824
  AI-날씨 유사도:     0.0036


## 4. 벡터 검색 구현

질문을 벡터로 변환하고, 문서 벡터들과의 유사도를 계산하여 가장 관련성 높은 문서를 검색합니다.

In [3]:
# vector_search  벡터로된 문서장들을 서로 코사이유사도를 이용해서 높은순으로 정렬
def vector_search(query, documents, doc_embeddings, top_k=2):
    """질의에 대한 벡터 유사도 검색을 수행합니다."""
    query_embeded = get_embedding(query)
    simularity = []
    for i, doc_emb in enumerate(doc_embeddings):
        sim = cosine_similarity(query_embeded,doc_emb)
        simularity.append((i,sim,documents[i]))
    simularity.sort(key=lambda x : x[1],reverse=True)  # 내림차순
    return simularity[:top_k]    

# 검색 테스트
queries = [
    "AI와 기계학습의 관계는?",
    "프로그래밍 언어 추천",
    "내일 비가 오나요?"
]

for query in queries:
    print(f"\n{'=' * 60}")
    print(f"검색 질의: '{query}'")
    print("=" * 60)
    results = vector_search(query, texts, embeddings)
    for rank, (idx, sim, doc) in enumerate(results, 1):
        print(f"  {rank}위: [유사도: {sim:.4f}] {doc}")


검색 질의: 'AI와 기계학습의 관계는?'
  1위: [유사도: 0.5643] 인공지능은 인간의 학습 능력을 컴퓨터로 구현한 기술이다.
  2위: [유사도: 0.4389] 머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야이다.

검색 질의: '프로그래밍 언어 추천'
  1위: [유사도: 0.4392] 파이썬은 데이터 과학에 널리 사용되는 프로그래밍 언어이다.
  2위: [유사도: 0.2198] 인공지능은 인간의 학습 능력을 컴퓨터로 구현한 기술이다.

검색 질의: '내일 비가 오나요?'
  1위: [유사도: 0.2694] 오늘 서울의 날씨는 맑고 기온은 25도이다.
  2위: [유사도: 0.0762] 머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야이다.


## 5. ChromaDB 벡터 스토어 구축

ChromaDB는 오픈소스 벡터 데이터베이스로, 설치와 사용이 간편하며 인메모리/로컬 모드를 지원합니다. RAG 시스템의 벡터 스토어로 활용합니다.

In [25]:
# AI/RAG 관련 한국어 문서 10개
documents = [
    "RAG는 검색 증강 생성의 약자로, LLM의 한계를 보완하는 기술이다.",
    "벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 데이터베이스이다.",
    "임베딩은 텍스트를 수치 벡터로 변환하는 과정으로, 의미적 유사성을 보존한다.",
    "LangChain은 LLM 기반 애플리케이션을 구축하기 위한 프레임워크이다.",
    "트랜스포머는 어텐션 메커니즘을 기반으로 한 딥러닝 아키텍처이다.",
    "프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기법이다.",
    "파인튜닝은 사전 학습된 모델을 특정 도메인에 맞게 추가 학습하는 과정이다.",
    "청킹은 긴 문서를 작은 단위로 분할하는 과정으로, RAG의 핵심 전처리 단계이다.",
    "코사인 유사도는 두 벡터 간의 각도를 측정하여 유사성을 평가하는 방법이다.",
    "지식 그래프는 엔티티와 관계를 그래프 구조로 표현하는 지식 표현 방식이다."
]

import chromadb
# 인메모리방식
chroma_client = chromadb.Client()
# 컬렉션 생성
collection = chroma_client.get_or_create_collection(name='rag_demo',metadata={'hnsw:space':'cosine'})
# 문서 추가
collection.add(
    ids = [f'doc_{i}' for i in range(len(documents))],
    documents=documents,
    metadatas=[ { 'soucrce':'rag_demo','chunk_id':i }   for i in range(len(documents)) ]
)

# 유사도 검색
query = 'Rag 시스템에서 문서를 어떻게 검색하나요?'
results = collection.query(query_texts=[query], n_results=2)
for doc,dis in zip(results['documents'][0],results['distances'][0]):
    print(f'{1-dis} | {doc}')


0.7829707860946655 | 청킹은 긴 문서를 작은 단위로 분할하는 과정으로, RAG의 핵심 전처리 단계이다.
0.696295976638794 | RAG는 검색 증강 생성의 약자로, LLM의 한계를 보완하는 기술이다.
